In [0]:
""" Utiities for Ingestion, transformation and enrichment """

import logging
from datetime import datetime
import json
from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window as W
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType


In [0]:
def get_logger(name="ecommerce_pipeline", level=logging.INFO):
    """Return a configured logger."""
    logger = logging.getLogger(name)
    if not logger.hasHandlers():
        logger.setLevel(level)
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        ))
        logger.addHandler(handler)
    return logger

In [0]:
logger = get_logger()

In [0]:
def parse_config_from_json(json_path):
    """Read a JSON file and return its contents as a dict."""
    
    try:
        with open(json_path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"JSON file not found: {json_path}")
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Failed to parse JSON '{json_path}': {e}")
    except Exception as e:
        raise RuntimeError(f"Failed to read JSON '{json_path}': {e}")

In [0]:
def get_value_from_dict(dictionary, key):
    """Recursively search for a key in a nested dictionary and return its value."""
    
    if not isinstance(dictionary, dict):
        raise TypeError(f"Input dictionary must be a dictionary, got {type(dictionary)}")
    if key in dictionary:
        return dictionary[key]
    for value in dictionary.values():
        if isinstance(value, dict):
            result = get_value_from_dict(value, key)
            if result is not None:
                return result
    return None



In [0]:
# --- File Readers ---- #

def json_reader(file_path: str, multi_line: bool = True) -> DataFrame:
    """ Read a JSON file into a DataFrame """
    try:
        logger.info("Reading JSON: %s (multi_line=%s)", file_path, multi_line)
        if multi_line:
            df = (
                spark.read.format("json")
                .option("multiLine", multi_line)
                .load(file_path)
            )
        else:
            df = spark.read.json(file_path)
        logger.info("JSON read complete – %d columns detected", len(df.columns))
        return df
    except Exception as e:
        if "Path does not exist" in str(e):
            raise FileNotFoundError(f"JSON file not found: {file_path}") from e
        raise RuntimeError(f"Failed to read JSON '{file_path}': {e}") from e


def csv_reader(file_path: str, delimiter: str = ",", header: bool = True) -> DataFrame:
    """ Read a CSV file into a DataFrame """
    try:
        logger.info("Reading CSV: %s (delimiter='%s', header=%s)", file_path, delimiter, header)
        df = (
            spark.read.format("csv")
            .options(header=header, delimiter=delimiter, escape='"')
            .load(file_path)
        )
        logger.info("CSV read complete – %d columns detected", len(df.columns))
        return df
    except Exception as e:
        if "Path does not exist" in str(e):
            raise FileNotFoundError(f"CSV file not found: {file_path}") from e
        raise RuntimeError(f"Failed to read CSV '{file_path}': {e}") from e


def excel_reader(file_path: str, sheet_name: str) -> DataFrame:
    """Read an Excel sheet into a DataFrame."""
    try:
        logger.info("Reading Excel: %s (sheet='%s')", file_path, sheet_name)
        df = (
            spark.read.format("excel")
            .option("headerRows", 1)
            .option("inferSchema", "true")
            .load(file_path, sheetName=sheet_name)
        )
        logger.info("Excel read complete – %d columns detected", len(df.columns))
        return df
    except Exception as e:
        if "Path does not exist" in str(e):
            raise FileNotFoundError(f"Excel file not found: {file_path}") from e
        raise RuntimeError(f"Failed to read Excel '{file_path}': {e}") from e

In [0]:
# ---- schema alias utils ----- #

def apply_schema_alias(df: DataFrame, schema_dict: dict) -> DataFrame:
    """Rename columns according to a mapping dictionary."""
    try:
        aliases = schema_dict
        logger.info("Applying %d column aliases", len(aliases))
        return df.select(
            [df[col].alias(aliases[col]) if col in aliases else df[col] for col in df.columns]
        )
    except Exception as e:
        logger.error("Failed to apply schema: %s", e)
        raise RuntimeError(f"Failed to apply schema: {e}") from e

In [0]:
# ---- Delta Table Writer ---- #

def delta_writer(
    df: DataFrame,
    table_name: str,
    mode: str = "overwrite",
    partition_by: list = None,
) -> None:
    """Write a DataFrame to a Delta table."""
    
    if df is None:
        raise ValueError("Cannot write a None DataFrame.")
    if not table_name or not table_name.strip():
        raise ValueError("table_name must be a non-empty string.")

    try:
        logger.info("Writing to %s (mode=%s)", table_name, mode)
        writer = (
            df.write
            .mode(mode)
            .option("overwriteSchema", "true")
            .format("delta")
        )
        if partition_by:
            writer = writer.partitionBy(*partition_by)
        writer.saveAsTable(table_name)
        logger.info("Write complete: %s", table_name)
    except Exception as e:
        raise RuntimeError(f"Failed to write Delta table '{table_name}': {e}") from e

In [0]:
def fill_defaults(df: DataFrame, default_values: dict) -> DataFrame:
    """Fill missing values in a DataFrame with default values."""
    if df is None:
        raise ValueError("Cannot fill defaults for a None DataFrame.")
    if not default_values:
        return df

    try:
        logger.info("Filling missing values with defaults: %s", default_values)
        return df.fillna(default_values)
    except Exception as e:
        logger.error("Failed to fill defaults: %s", e)
        raise RuntimeError(f"Failed to fill defaults: {e}") from e

In [0]:
def apply_schema(df: DataFrame, schema: StructType) -> DataFrame:
    """Apply a schema to a DataFrame."""
    if df is None:
        raise ValueError("Cannot apply schema to a None DataFrame.")
    if not schema:
        raise ValueError("Schema must be a non-empty StructType.")
    try:
        result = df.select(*schema.names)
        logger.info("Schema applied: %s", schema.names)
        return result
    except Exception as e:
        logger.error("Failed to apply schema: %s", e)
        raise RuntimeError(f"Failed to apply schema: {e}") from e

In [0]:

def remove_nulls_for_key(df: DataFrame, primary_key) -> DataFrame:
    """Remove rows with nulls in the primary key(s) and log the count.
    primary_key: str or list of str
    """
    try:
        if isinstance(primary_key, str):
            keys = [primary_key]
        elif isinstance(primary_key, list) and all(isinstance(k, str) for k in primary_key):
            keys = primary_key
        else:
            raise TypeError("primary_key must be a str or list of str.")

        not_null_condition = [F.col(k).isNotNull() for k in keys]
        if len(keys) == 1:
            filtered_df = df.filter(not_null_condition[0])
        else:
            filtered_df = df.filter(reduce(lambda a, b: a & b, not_null_condition))
        null_count = df.count() - filtered_df.count()
        if null_count > 0:
            logger.info("Removed %d rows with null in %s", null_count, keys)
        return filtered_df
    except Exception as e:
        logger.error("Failed to remove nulls for key(s) %s: %s", primary_key, e)
        raise RuntimeError(f"Failed to remove nulls for key(s) '{primary_key}': {e}") from e

In [0]:
def parse_dates(df: DataFrame, date_formats: dict) -> DataFrame:
    """
    Parse date columns with specified formats, handling mixed formats.
    date_formats: dict mapping column name to format string (e.g. {'order_date': 'd/M/yyyy'})
    """
    for col, fmt in date_formats.items():
        df = df.withColumn(
            col,
            F.coalesce(
                F.to_date(F.col(col), fmt),
                F.to_date(F.col(col))
            )
        )
    return df

In [0]:
def parse_struct_schema(schema_str: str) -> StructType:
    """Parse a StructType string from JSON config into a PySpark StructType."""
    try:
        schema = eval(schema_str, {"__builtins__": {}}, T.__dict__)
        if not isinstance(schema, StructType):
            raise ValueError(f"Expected StructType, got {type(schema).__name__}")
        logger.info("Parsed schema with %d fields: %s", len(schema.fields), schema.names)
        return schema
    except Exception as e:
        logger.error("Failed to parse struct schema: %s", e)
        raise RuntimeError(f"Failed to parse struct schema: {e}") from e